# DAP-seq Report Notebook

Use this notebook to query the shared pipeline database, filter runs, and regenerate the HTML/TSV report for any subset of samples.

**Setup:** Copy this notebook to your own directory before editing — don't modify the template in the pipeline repo.

In [ ]:
import sqlite3
import sys
import pandas as pd
from pathlib import Path

# Path to the shared pipeline database
DB_PATH = "/path/to/pipeline/pipeline_db.db"

# Make report.py helpers importable
PIPELINE_DIR = "/path/to/pipeline"
sys.path.insert(0, str(Path(PIPELINE_DIR) / "workflow" / "scripts"))
from report import COLS, write_html, logo_to_base64

## Load the database

In [ ]:
con = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM pipeline_runs", con)
con.close()

print(f"{len(df)} rows, {df['output_dir'].nunique()} unique runs")
df.head()

## Explore

In [ ]:
# All unique runs in the database
df[["output_dir", "run_date", "genome_ref"]].drop_duplicates().sort_values("run_date", ascending=False)

In [ ]:
# Numeric summary of QC columns
numeric_cols = ["total_frags", "clean_reads", "filtered_reads", "peak#", "min5fold_peak#", "FRiP_score"]
df[numeric_cols].apply(pd.to_numeric, errors="coerce").describe()

## Filter

Edit the cell below to select the rows you want in the report.
Each example is independent — combine them with `&` as needed.

In [ ]:
filtered = df.copy()

# --- filter by specific output directory ---
# filtered = filtered[filtered["output_dir"] == "/scratch/myproject/run1"]

# --- filter by sample name (substring match) ---
# filtered = filtered[filtered["sample"].str.contains("TF1")]

# --- filter by date range ---
# filtered = filtered[filtered["run_date"] >= "2025-01-01"]
# filtered = filtered[filtered["run_date"].between("2025-01-01", "2025-06-30")]

# --- filter by genome reference ---
# filtered = filtered[filtered["genome_ref"].str.contains("GRCh38")]

# --- filter by minimum FRiP score ---
# filtered = filtered[pd.to_numeric(filtered["FRiP_score"], errors="coerce") >= 5.0]

# --- filter to treatment samples only (exclude controls) ---
# filtered = filtered[filtered["is_treatment"] == "True"]

print(f"{len(filtered)} rows selected")
filtered[["sample", "output_dir", "run_date", "peak#", "FRiP_score"]]

## Generate report

`generate_report(df, out_dir)` writes `report.tsv` and `report.html` to the directory you specify.
It tries to load motif logos from the original output directories; logos are silently skipped if the path no longer exists.

In [ ]:
# Column name mapping: DB stores underscored names, report.py expects spaced names
_DB_TO_REPORT = {
    "min5fold_peak#": "min5fold peak#",
    "max_peak_score": "max peak score",
    "peak_reads#":    "peak reads#",
}

def generate_report(df, out_dir):
    """Write report.tsv and report.html for the given filtered DataFrame."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Rename DB column names back to the names report.py expects
    df = df.rename(columns=_DB_TO_REPORT)

    rows = df.to_dict(orient="records")

    # Try to load motif logos from the original output directories
    logo_b64_map = {}
    for row in rows:
        sample     = row["sample"]
        output_dir = row["output_dir"]
        logo_path  = Path(output_dir) / "meme" / sample / "summits" / "logo1.png"
        logo_b64_map[sample] = logo_to_base64(str(logo_path))

    # Write TSV
    tsv_path = out_dir / "report.tsv"
    with open(tsv_path, "w") as fh:
        fh.write("\t".join(COLS) + "\n")
        for row in rows:
            fh.write("\t".join(str(row.get(c, "NA")) for c in COLS) + "\n")

    # Write HTML
    html_path = out_dir / "report.html"
    write_html(rows, logo_b64_map, str(html_path))

    print(f"Written: {tsv_path}")
    print(f"Written: {html_path}")

In [ ]:
# Edit the output directory to wherever you want the report files
generate_report(filtered, "/path/to/my/report_output")